In [ ]:
import pandas as pd
import datetime
import numpy as np
import math
import random
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle

In [ ]:
# import os
# os.getcwd()
merged_data = pd.read_csv(r"D:\FPL\fpl-optimization\data\Fantasy-Premier-League\data\cleaned_merged_seasons1.csv", index_col=False, dtype='unicode')
# merged2122_df = pd.read_csv("datasets\merged_gw2122.csv")

In [ ]:
merged_data.head()

In [ ]:
merged_data.dtypes

In [ ]:
def plot_nas(df: pd.DataFrame):
    if df.isnull().sum().sum() != 0:
        na_df = (df.isnull().sum() / len(df)) * 100      
        na_df = na_df.drop(na_df[na_df == 0].index).sort_values(ascending=False)
        missing_data = pd.DataFrame({'Missing Ratio %' :na_df})
        missing_data.plot(kind = "barh")
        plt.show()
    else:
        print('No NAs found')
plot_nas(merged_data)
plot_width, plot_height = (20,24)
plt.rcParams['figure.figsize'] = (plot_width,plot_height)

In [ ]:
merged_data[merged_data.isnull().any(axis=1)]

In [ ]:
# teams = merged_data.team.unique()
# understat_dfs = {team: pd.read_csv(f"datasets\\understat\\understat_{team}_.csv") for team in teams}

In [ ]:
# # Load the fixtures file
# fixtures_data = pd.read_csv("datasets\\fixtures.csv")

# # Load the master team list containing id and name
# team_list = pd.read_csv("datasets\\teams.csv")

In [ ]:
# # Encode team names to nominal ids in fixtures_data
# team_id_map = {team_id: team_name for team_id, team_name in zip(team_list['id'],team_list['name'])}
# fixtures_data['team_a'] = fixtures_data['team_a'].apply(lambda x: team_id_map.get(x, x))
# fixtures_data['team_h'] = fixtures_data['team_h'].apply(lambda x: team_id_map.get(x, x))

In [ ]:
# fixtures_data.head()

In [ ]:
# merged_data['xG'] = None
# merged_data['xGA'] = None
# merged_data['npxG'] = None
# merged_data['npxGA'] = None
# merged_data['deep'] = None
# merged_data['deep_allowed'] = None
# merged_data['scored'] = None
# merged_data['missed'] = None
# merged_data['xpts'] = None
# merged_data['pts'] = None
# merged_data['opp_xG'] = None
# merged_data['opp_xGA'] = None
# merged_data['opp_npxG'] = None
# merged_data['opp_npxGA'] = None
# merged_data['opp_deep'] = None
# merged_data['opp_deep_allowed'] = None
# merged_data['opp_scored'] = None
# merged_data['opp_missed'] = None
# merged_data['opp_xpts'] = None
# merged_data['opp_pts'] = None

In [ ]:
# Merge understat and player data by matching kickoff_date

# for _, player in merged_data.iterrows():
#     team_understat = understat_dfs[player['team']]
#     opp_understat = understat_dfs[player['opp_team_name']]
#     date = player['kickoff_date']
#     player['xG'] = team_understat[team_understat['date']==date]['xG'].values[0]
#     player['xGA'] = team_understat[team_understat['date']==date]['xGA'].values[0]
#     player['npxG'] = team_understat[team_understat['date']==date]['npxG'].values[0]
#     player['npxGA'] = team_understat[team_understat['date']==date]['npxGA'].values[0]
#     player['deep'] = team_understat[team_understat['date']==date]['deep'].values[0]
#     player['deep_allowed'] = team_understat[team_understat['date']==date]['deep_allowed'].values[0]
#     player['scored'] = team_understat[team_understat['date']==date]['scored'].values[0]
#     player['missed'] = team_understat[team_understat['date']==date]['missed'].values[0]
#     player['xpts'] = team_understat[team_understat['date']==date]['xpts'].values[0]
#     player['pts'] = team_understat[team_understat['date']==date]['pts'].values[0]
#     player['opp_xG'] = opp_understat[opp_understat['date']==date]['xG'].values[0]
#     player['opp_xGA'] = opp_understat[opp_understat['date']==date]['xGA'].values[0]
#     player['opp_npxG'] = opp_understat[opp_understat['date']==date]['npxG'].values[0]
#     player['opp_npxGA'] = opp_understat[opp_understat['date']==date]['npxGA'].values[0]
#     player['opp_deep'] = opp_understat[opp_understat['date']==date]['deep'].values[0]
#     player['opp_deep_allowed'] = opp_understat[opp_understat['date']==date]['deep_allowed'].values[0]
#     player['opp_scored'] = opp_understat[opp_understat['date']==date]['scored'].values[0]
#     player['opp_missed'] = opp_understat[opp_understat['date']==date]['missed'].values[0]
#     player['opp_xpts'] = opp_understat[opp_understat['date']==date]['xpts'].values[0]
#     player['opp_pts'] = opp_understat[opp_understat['date']==date]['pts'].values[0]



# for team in teams:
#     understat_df = understat_dfs[team]

#     # Convert understat date column to match the format in fixtures_data
#     understat_df['date'] = pd.to_datetime(understat_df['date']).dt.strftime('%Y-%m-%d')
#     fixtures_data['kickoff_time'] = pd.to_datetime(fixtures_data['kickoff_time']).dt.strftime('%Y-%m-%d')
#     fixture_info = fixtures_data[
#         (fixtures_data['team_a'] == team) | (fixtures_data['team_h'] == team)
#     ]
#     # Merge understat_df with fixtures_data based on kickoff_time and date
#     merged_df = pd.merge(understat_df, fixture_info[['kickoff_time', 'event', 'team_h_difficulty', 'team_a_difficulty']],
#                          left_on='date', right_on='kickoff_time', how='left')
#     merged_df['team_difficulty'] = float("nan")
#     for _,row in merged_df.iterrows():
#         if row['h_a']=='a':
#             row['team_difficulty'] = row['team_a_difficulty']
#         else:
#             row['team_difficulty'] = row['team_h_difficulty']
#         merged_df.at[_,'team_difficulty'] = row['team_difficulty']
#     # Drop the redundant kickoff_time column from the merged_df
#     merged_df.drop(['kickoff_time','team_h_difficulty', 'team_a_difficulty'], axis=1, inplace=True)
#     merged_df.rename(columns={'event': 'GW'}, inplace=True)

#     # Save the merged data back to the understat file for the team
# #     merged_df.to_csv(f'understat_{team}.csv', index=False)
#     understat_dfs[team] = merged_df

In [ ]:
merged_data['fdr_team'] = merged_data['fdr_team'].astype(float)
merged_data['fdr_opp_team'] = merged_data['fdr_opp_team'].astype(float)


In [ ]:
# Creating modified variables
merged_data['fdr_team'] = merged_data['fdr_team'].astype(float)
merged_data['fdr_opp_team'] = merged_data['fdr_opp_team'].astype(float)
# merged_data['transfers_in'] = merged_data['transfers_in'].astype(float)
# merged_data['transfers_out'] = merged_data['transfers_out'].astype(float)

merged_data['fdr_net'] = merged_data['fdr_team'] - merged_data['fdr_opp_team']
# merged_data['transfer_net'] = merged_data['transfers_in'] - merged_data['transfers_out']
# merged_data['xGD'] = merged_data['xG'] - merged_data['xGA']
# merged_data['npxGD'] = merged_data['npxG'] - merged_data['npxGA']

In [ ]:
# Create a new column to store the match count
merged_data['match_count'] = merged_data.groupby('name').cumcount() + 1

In [ ]:
merged_data.columns

In [ ]:
# Create a function to add previous gameweek data for each player
def add_previous_match_data(player_data, previous_gw_data, gw_number):
    prefix = f"{gw_number}_"
    player_data[prefix + "total_points"] = previous_gw_data['total_points'].values[0]
    player_data[prefix + "assists"] = previous_gw_data['assists'].values[0]
    player_data[prefix + "bonus"] = previous_gw_data['bonus'].values[0]
    player_data[prefix + "bps"] = previous_gw_data['bps'].values[0]
    player_data[prefix + "clean_sheets"] = previous_gw_data['clean_sheets'].values[0]
    player_data[prefix + "creativity"] = previous_gw_data['creativity'].values[0]
    player_data[prefix + "goals_conceded"] = previous_gw_data['goals_conceded'].values[0]
    player_data[prefix + "goals_scored"] = previous_gw_data['goals_scored'].values[0]
    player_data[prefix + "ict_index"] = previous_gw_data['ict_index'].values[0]
    player_data[prefix + "influence"] = previous_gw_data['influence'].values[0]
    player_data[prefix + "minutes"] = previous_gw_data['minutes'].values[0]
#     player_data[prefix + "opponent_team"] = previous_gw_data['opponent_team'].values[0]
    player_data[prefix + "opp_team_name"] = previous_gw_data['opponent_team'].values[0]
    player_data[prefix + "own_goals"] = previous_gw_data['own_goals'].values[0]
    player_data[prefix + "penalties_missed"] = previous_gw_data['penalties_missed'].values[0]
    player_data[prefix + "penalties_saved"] = previous_gw_data['penalties_saved'].values[0]
    player_data[prefix + "red_cards"] = previous_gw_data['red_cards'].values[0]
#     player_data[prefix + "round"] = previous_gw_data['round'].values[0]
    player_data[prefix + "saves"] = previous_gw_data['saves'].values[0]
    player_data[prefix + "selected"] = previous_gw_data['selected'].values[0]
    player_data[prefix + "threat"] = previous_gw_data['threat'].values[0]
    player_data[prefix + "yellow_cards"] = previous_gw_data['yellow_cards'].values[0]
#     player_data[prefix + "goals_scored_team"] = previous_gw_data['goals_scored_team']
#     player_data[prefix + "goals_conceded_team"] = previous_gw_data['goals_conceded_team']
    player_data[prefix + "was_home"] = previous_gw_data['was_home'].values[0]
    if player_data[prefix + "was_home"]:
        player_data[prefix + "goals_scored_team"] = previous_gw_data['team_h_score'].values[0]
        player_data[prefix + "goals_conceded_team"] = previous_gw_data['team_a_score'].values[0]
    else:
        player_data[prefix + "goals_scored_team"] = previous_gw_data['team_a_score'].values[0]
        player_data[prefix + "goals_conceded_team"] = previous_gw_data['team_h_score'].values[0]
    # player_data[prefix + "goals_scored_team"] = previous_gw_data['goals_scored_team'].values[0]
    # player_data[prefix + "goals_conceded_team"] = previous_gw_data['goals_conceded_team'].values[0]
    player_data[prefix + "fdr_net"] = previous_gw_data['fdr_net'].values[0]
    player_data[prefix + "transfers_net"] = previous_gw_data['transfers_net'].values[0]
    # player_data[prefix + "transfers_out"] = previous_gw_data['transfers_out'].values[0]
    player_data[prefix + "value"] = previous_gw_data['value'].values[0]
#     player_data[prefix + "xG"] = previous_gw_data['xG'].values[0]
#     player_data[prefix + "xGA"] = previous_gw_data['xGA'].values[0]
# #     player_data[prefix + "xGD"] = previous_gw_data['xGD'].values[0]
#     player_data[prefix + "npxG"] = previous_gw_data['npxG'].values[0]
#     player_data[prefix + "npxGA"] = previous_gw_data['npxGA'].values[0]
# #     player_data[prefix + "npxGD"] = previous_gw_data['npxGD'].values[0]
#     player_data[prefix + "deep"] = previous_gw_data['deep'].values[0]
#     player_data[prefix + "deep_allowed"] = previous_gw_data['deep_allowed'].values[0]
# #     player_data[prefix + "scored"] = previous_gw_data['scored'].values[0]
# #     player_data[prefix + "missed"] = previous_gw_data['missed'].values[0]
#     player_data[prefix + "xpts"] = previous_gw_data['xpts'].values[0]
#     player_data[prefix + "pts"] = previous_gw_data['pts'].values[0]
# #     player_data[prefix + "opp_xG"] = previous_gw_data['opp_xG'].values[0]
# #     player_data[prefix + "opp_xGA"] = previous_gw_data['opp_xGA'].values[0]
# #     player_data[prefix + "opp_npxG"] = previous_gw_data['opp_npxG'].values[0]
# #     player_data[prefix + "opp_npxGA"] = previous_gw_data['opp_npxGA'].values[0]
# #     player_data[prefix + "opp_deep"] = previous_gw_data['opp_deep'].values[0]
# #     player_data[prefix + "opp_deep_allowed"] = previous_gw_data['opp_deep_allowed'].values[0]
# #     player_data[prefix + "opp_scored"] = previous_gw_data['opp_scored'].values[0]
# #     player_data[prefix + "opp_missed"] = previous_gw_data['opp_missed'].values[0]
#     player_data[prefix + "opp_xpts"] = previous_gw_data['opp_xpts'].values[0]
#     player_data[prefix + "opp_pts"] = previous_gw_data['opp_pts'].values[0]

# Create a list to hold each player's timeseries data
timeseries_data_list = []
# Iterate through each player in the merged dataset and add previous gameweek data
for _, player in merged_data.iterrows():
    player_data = player[['name','team_x','position','match_count','season_x','GW','total_points','opp_team_name','transfers_in','transfers_out','selected','value','was_home','fdr_team','fdr_opp_team']].copy()
    for m in range(1,5):
        previous_match = player['match_count'] - m
        if previous_match < 1:
            break
        previous_match_data = merged_data[
            (merged_data['name'] == player['name']) &
            (merged_data['match_count'] == previous_match)
        ]
        if previous_match_data.shape[0]>0:
            add_previous_match_data(player_data, previous_match_data, m)

    timeseries_data_list.append(player_data)

#     team = player['team']
#     for gw in range(1, 5):
#         if team in understat_dfs:
#             team_understat_df = understat_dfs[team]
#             previous_gw = player['GW'] - gw
#             if previous_gw < 1:
#                 break
#             previous_gw_data = team_understat_df[
#                 team_understat_df['GW'] == previous_gw
#             ]
#             prefix = f"{gw}_"
#             player_data[prefix + "xG"] = previous_gw_data['xG'].values[0]
#             player_data[prefix + "xGA"] = previous_gw_data['xGA'].values[0]
#             player_data[prefix + "npxG"] = previous_gw_data['npxG'].values[0]
#             player_data[prefix + "npxGA"] = previous_gw_data['npxGA'].values[0]
#             player_data[prefix + "deep"] = previous_gw_data['deep'].values[0]
#             player_data[prefix + "deep_allowed"] = previous_gw_data['deep_allowed'].values[0]
#             player_data[prefix + "scored"] = previous_gw_data['scored'].values[0]
#             player_data[prefix + "missed"] = previous_gw_data['missed'].values[0]
#             player_data[prefix + "xpts"] = previous_gw_data['xpts'].values[0]
#             player_data[prefix + "wins"] = previous_gw_data['wins'].values[0]
#             player_data[prefix + "draws"] = previous_gw_data['draws'].values[0]
#             player_data[prefix + "loses"] = previous_gw_data['loses'].values[0]
#             player_data[prefix + "npxGD"] = previous_gw_data['npxGD'].values[0]
#             player_data[prefix + "difficulty"] = previous_gw_data['team_difficulty'].values[0]

#     for gw in range(1, 5):
#         if team_id_map[player_data[f'{gw}_opponent_team']] in understat_dfs:
#             team_understat_df = understat_dfs[team_id_map[player_data[f'{gw}_opponent_team']]]
#             previous_gw = player['GW'] - gw
#             if previous_gw < 1:
#                 break
#             previous_gw_data = team_understat_df[
#                 team_understat_df['GW'] == previous_gw
#             ]
#             prefix = f"{gw}_opp_"
#             player_data[prefix + "xG"] = previous_gw_data['xG'].values[0]
#             player_data[prefix + "xGA"] = previous_gw_data['xGA'].values[0]
#             player_data[prefix + "npxG"] = previous_gw_data['npxG'].values[0]
#             player_data[prefix + "npxGA"] = previous_gw_data['npxGA'].values[0]
#             player_data[prefix + "deep"] = previous_gw_data['deep'].values[0]
#             player_data[prefix + "deep_allowed"] = previous_gw_data['deep_allowed'].values[0]
#             player_data[prefix + "scored"] = previous_gw_data['scored'].values[0]
#             player_data[prefix + "missed"] = previous_gw_data['missed'].values[0]
#             player_data[prefix + "xpts"] = previous_gw_data['xpts'].values[0]
#             player_data[prefix + "wins"] = previous_gw_data['wins'].values[0]
#             player_data[prefix + "draws"] = previous_gw_data['draws'].values[0]
#             player_data[prefix + "loses"] = previous_gw_data['loses'].values[0]
#             player_data[prefix + "npxGD"] = previous_gw_data['npxGD'].values[0]
#             player_data[prefix + "difficulty"] = previous_gw_data['team_difficulty'].values[0]
    


In [ ]:
# Optimized function to add previous gameweek data for each player
def add_previous_match_data(player_data, previous_gw_data, gw_number):
    prefix = f"{gw_number}_"
    cols_to_copy = ['assists', 'bonus', 'bps', 'clean_sheets', 'creativity', 'goals_conceded', 'goals_scored', 
                    'ict_index', 'influence', 'minutes', 'opp_team_name', 'own_goals', 'penalties_missed', 
                    'penalties_saved', 'red_cards', 'saves', 'selected', 'threat', 'total_points', 
                    'transfers_balance', 'value', 'was_home', 'yellow_cards', 'fdr_net']
    
    # Vectorize the assignment of multiple columns
    for col in cols_to_copy:
        player_data[f"{prefix}{col}"] = previous_gw_data[col].values[0]
    
    # Efficient handling of team scores
    if player_data[prefix + "was_home"]:
        player_data[f"{prefix}goals_scored_team"] = previous_gw_data['team_h_score'].values[0]
        player_data[f"{prefix}goals_conceded_team"] = previous_gw_data['team_a_score'].values[0]
    else:
        player_data[f"{prefix}goals_scored_team"] = previous_gw_data['team_a_score'].values[0]
        player_data[f"{prefix}goals_conceded_team"] = previous_gw_data['team_h_score'].values[0]

# Pre-filter the relevant columns once
cols = ['season_x', 'name', 'position', 'team_x', 'opp_team_name',
       'total_points', 'was_home', 'GW', 'xP', 'fdr_net', 'match_count']

# Create a dictionary to hold timeseries data
timeseries_data_list = []

# Pre-filter the merged_data by name to avoid redundant filtering
grouped_data = merged_data.groupby('name')

# Iterate through each player
for name, player_group in grouped_data:
    # Iterate through each match the player played
    for _, player in player_group.iterrows():
        player_data = player[cols].copy()
        match_count = player['match_count']
        
        # Add previous matches (up to 4 previous gameweeks)
        for m in range(1, 5):
            previous_match = match_count - m
            if previous_match < 1:
                break
            
            previous_match_data = player_group[player_group['match_count'] == previous_match]
            
            if not previous_match_data.empty:
                add_previous_match_data(player_data, previous_match_data, m)
        
        timeseries_data_list.append(player_data)

# Convert the list to a DataFrame for further analysis or output
timeseries_data_df = pd.DataFrame(timeseries_data_list)
timeseries_data_df.head()

In [ ]:
player_data

In [ ]:
# Convert the list of dictionaries to a DataFrame
timeseries_data = pd.DataFrame(timeseries_data_list)

# Reset the index of the new timeseries dataset
timeseries_data.reset_index(drop=True, inplace=True)

# Save the timeseries dataset to a new CSV file
timeseries_data.to_csv(r'data/all_players_timeseries_dataset.csv', index=False)

In [ ]:
timeseries_data.tail()

In [ ]:
timeseries_data.iloc[5000:5005,100:110]

In [ ]:
# Vector approach

# Assuming merged_data is already defined
merged_data_copy = merged_data.copy()  # Create a copy of the original DataFrame
# Create a shifted DataFrame with previous match data for each player
shifted_data = merged_data_copy[['name','team_x','position','match_count','season_x','GW','total_points','opp_team_name','transfers_net','selected','value','was_home','fdr_team','fdr_opp_team']].copy() 
for m in range(1, 5):
    shifted = merged_data_copy.copy() 
    shifted['match_count'] += m  # Shift match_count to simulate previous matches
    shifted = shifted.add_prefix(f'{m}')  # Rename columns to distinguish them
    # Merge the shifted data back with the original data
    merged_data_copy = merged_data_copy.merge(
        shifted,
        left_on=['name', 'match_count'],
        right_on=[f'{m}_name', f'{m}_match_count'],
        how='left'
    )

    # Drop the unnecessary columns from the merged result
    merged_data_copy.drop(
        [f'{m}_season', f'{m}_name', f'{m}_match_count'],
        axis=1,
        inplace=True
    )

In [ ]:
# Create a function to add previous gameweek data for each player
def add_previous_match_data(player_data, previous_gw_data, gw_number):
    prefix = f"{gw_number}_"
    player_data.loc[:, [prefix + col for col in previous_gw_data.columns]] = previous_gw_data.values
    
    # Update 'was_home' and 'goals_scored_team'/'goals_conceded_team' columns
    player_data.loc[:, [prefix + col for col in ['was_home', 'goals_scored_team', 'goals_conceded_team']]] = np.where(
        player_data[prefix + 'was_home'],
        player_data[['team_h_score', 'team_a_score']].values.T,
        player_data[['team_a_score', 'team_h_score']].values.T
    )
    
    # Update 'transfers_in' and 'transfers_out' columns
    player_data.loc[:, [prefix + col for col in ['transfers_in', 'transfers_out']]] = player_data[['transfers_in', 'transfers_out']].values.T
    
    # Update 'value' column
    player_data.loc[:, [prefix + 'value']] = player_data['value'].values.reshape(-1, 1)

# Create a list to store the timeseries data
timeseries_data_list = []

# Iterate over the merged_data DataFrame in chunks
chunk_size = 100000
for i in range(0, len(merged_data), chunk_size):
    chunk = merged_data.iloc[i:i+chunk_size]
    
    # Create a new DataFrame to store the timeseries data for the chunk
    chunk_timeseries_data = pd.DataFrame(columns=['name', 'team_x', 'position', 'match_count', 'season_x', 'GW', 'total_points', 'opp_team_name', 'transfers_in', 'transfers_out', 'selected', 'value', 'was_home', 'fdr_team', 'fdr_opp_team'])
    
    # Iterate over the players in the chunk
    for _, player in chunk.iterrows():
        player_data = player[['name', 'team_x', 'position', 'match_count', 'season_x', 'GW', 'total_points', 'opp_team_name', 'transfers_in', 'transfers_out', 'selected', 'value', 'was_home', 'fdr_team', 'fdr_opp_team']].copy()
        
        # Find the previous match data for the player
        previous_match_data = merged_data[(merged_data['name'] == player['name']) & (merged_data['match_count'] < player['match_count'])].sort_values('match_count', ascending=False).iloc[:4]
        
        # Add the previous match data to the player_data DataFrame
        for m, data in previous_match_data.iterrows():
            add_previous_match_data(player_data, data, m+1)
        
        # Append the player_data DataFrame to the chunk_timeseries_data DataFrame
        chunk_timeseries_data = pd.concat([chunk_timeseries_data, player_data], ignore_index=True)
    
    # Append the chunk_timeseries_data DataFrame to the timeseries_data_list
    timeseries_data_list.extend(chunk_timeseries_data.values)

In [ ]:
merged_data_copy.tail()